# VIVO Backgammon — Entrenar la red en GPU (Colab)

Red wide-net (198→256→128→64→1, tanh), idéntica a la del navegador. El motor de backgammon está portado a Python para que la red juegue por las MISMAS reglas.

**No tienes que subir nada.** Las celdas 0 y 1 se bajan los archivos del repositorio automáticamente.

**Qué hacer (solo 3 clics):**
1. Menú superior → **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU** → Guardar.
2. Menú superior → **Entorno de ejecución → Ejecutar todo**.
3. Espera. Al terminar, la última celda te descarga `model_weights.json`.

**Salida:** `public/model_weights.json` en el formato EXACTO que carga el navegador VIVO (sin tocar el frontend).

In [ ]:
# === CELDA 0: instala TensorFlow y comprueba la GPU ===
import subprocess, sys, os, json

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow', 'numpy'])
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPU disponible:', gpus)
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print('OK: entrenamiento en GPU.')
else:
    print('AVISO: no hay GPU -> sera lento (CPU).')
os.makedirs('public', exist_ok=True)

In [ ]:
# === CELDA 1: descarga los archivos del repo (no subas nada manualmente) ===
BRANCH = '260816-gpu-train'
BASE = f'https://raw.githubusercontent.com/Pirzl/Backgammon-AR-Pro/{BRANCH}/colab'
subprocess.check_call(['curl', '-L', '-o', 'bg_engine.py', f'{BASE}/bg_engine.py'])
subprocess.check_call(['curl', '-L', '-o', 'bg_net.py',    f'{BASE}/bg_net.py'])
assert os.path.exists('bg_engine.py') and os.path.exists('bg_net.py'), 'fallo la descarga'
print('Archivos listos en', os.getcwd())

In [ ]:
# === CELDA 2: entrena (TD(0) vs heuristico, en GPU) ===
# Primer arranque conservador: 20000 partidas para validar que APRENDE (rate sube).
# Si ves 'rate' subir por encima de ~0.5 en las evals, ya esta aprendiendo y
# podemos subir --games a 1-2 millones en corridas posteriores (reanudando con --weights).
import bg_net as net
cmd = [
    sys.executable, 'bg_net.py',
    '--games', '20000',
    '--opponent', 'heuristic',
    '--label', 'td0',
    '--exploration', '0.15',
    '--max-moves', '400',
    '--epochs', '3',
    '--eval-every', '200',
    '--eval-games', '100',
    '--save-every', '50',
    '--stop-rate', '0.55',
    '--stop-streak', '2',
    '--out', 'public/model_weights.json',
    '--seed', '1',
]
print('Arrancando entrenamiento...')
subprocess.check_call(cmd)

In [ ]:
# === CELDA 3: verificacion de colapso (Python puro, sin TF) ===
# Lee public/model_weights.json y hace forward-pass sobre posiciones variadas.
# std < 1e-3 -> COLLAPSED (la red no reacciona). >50% unidades muertas -> WARNING.
import math, random
import bg_engine as bg
raw = json.load(open('public/model_weights.json'))
w = raw['weights']
W1,b1,W2,b2,W3,b3,W4,b4 = [l['data'] for l in w]
def mat(M, x, b):
    return [b[j] + sum(x[i]*M[i][j] for i in range(len(x))) for j in range(len(b))]
def fwd(fv):
    h1 = [max(0,v) for v in mat(W1, fv, b1)]
    h2 = [max(0,v) for v in mat(W2, h1, b2)]
    h3 = [max(0,v) for v in mat(W3, h2, b3)]
    y  = [math.tanh(v) for v in mat(W4, h3, b4)]
    return y[0], (h1,h2,h3)
rng = random.Random(1)
preds = []; hs = ([],[],[])
for t in ['white','black']:
    y,h = fwd(bg.encode_board(bg.INITIAL_BOARD, t)); preds.append(y)
    for k in range(8):
        b = [0]*30
        for _ in range(15): b[rng.randint(1,24)] += 1
        for _ in range(15): b[rng.randint(1,24)] -= 1
        y,h = fwd(bg.encode_board(b, t)); preds.append(y)
        for li,hh in enumerate(h): hs[li].append(hh)
mean = sum(preds)/len(preds)
std = math.sqrt(sum((p-mean)**2 for p in preds)/len(preds))
print('preds=', [round(p,3) for p in preds])
print(f'mean={mean:.4f} std={std:.4f} min={min(preds):.3f} max={max(preds):.3f}')
dead = []
for li,st in enumerate(hs):
    n = len(st[0]); d = sum(1 for u in range(n) if all(abs(st[k][u])<1e-9 for k in range(len(st))))
    dead.append(d/n)
print('dead units:', [f'{x*100:.0f}%' for x in dead])
if std < 1e-3:
    print('VEREDICTO: COLLAPSED (salida no reacciona)')
elif any(x > 0.5 for x in dead):
    print('VEREDICTO: WARNING (>50% unidades muertas)')
else:
    print('VEREDICTO: ALIVE')

In [ ]:
# === CELDA 4: descarga los pesos entrenados ===
from google.colab import files
files.download('public/model_weights.json')
# Llevalo a: E:/Proyecto/BACKGAMMON/BACKGAMMON-VIVO - copia/public/model_weights.json